# B1 · Carga / alineación / crop

**Spec:** [`docs/spec_B1_codex_load_align_crop.md`](../docs/spec_B1_codex_load_align_crop.md)  |  **Bloque:** B · Preparación  |  **Run de este set:** `ROXs12b_realigned`

Carga el cubo, centra y recorta al campo de interés.

| | |
|---|---|
| **Entrada** | `cube_telcorr.fits` |
| **Salida (QC/productos)** | `stages/stage01_qc.json` |
| **Consume aguas abajo** | B2, B3, C1 |


## Qué hace B1 y por qué importa

B1 **carga** el cubo reducido, lo **centra** en la estrella primaria y lo **recorta** a una ventana de 170 px → el cubo de trabajo para toda la extracción aguas abajo.

Es una **migración por equivalencia**: la lógica venía del notebook histórico `01_load_align_crop.ipynb` y se movió a `musepipe/stages/stage01_align.py` reproduciendo los productos numéricamente, y solo después se extendió (propagar STAT, registrar shifts, `entry_point`).

**En este run:** `n_cubes = 1` — la alineación entre exposiciones ya se hizo en A1 (plan B, OFFSET_LIST manual), así que el **shift de B1 es 0**; B1 solo centra y recorta el cubo combinado.

**El nexo con M5 (ruido):** B1 propaga la extensión STAT por las mismas transformaciones (kernel `bilinear_kernel_squared`) y registra el **factor de covarianza espacial box3 ≈ 6.4×**. Ese es el remuestreo del cubo que correlaciona el ruido — la raíz física de que el STAT subestime (M5) y de la inflación de apertura (G1). El shift subpixel (spline orden 3) de la alineación es el origen cuantificado en [`docs/noise_model.md`](../docs/noise_model.md) §4.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python -c "from musepipe.stages.stage01_align import run_stage01; run_stage01('$RUN')"
```

Ligero (segundos).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage01_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -c "from musepipe.stages.stage01_align import run_stage01; run_stage01(\'$RUN\')"'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage01_qc.json', RUN_ID)
nb.show(qc, keys=['centering_method', 'spatial_shift_mode', 'crop', 'n_cubes', 'covariance_factor_box3', 'finite_fraction'], title='B1')


## Los términos de este QC, en físico

| Término | Qué es | Por qué importa |
|---|---|---|
| `centering_method` / `star_centers` | Cómo se localizó la primaria en cada cubo para ponerlos todos en el mismo sitio. | Un centrado malo desplaza al compañero respecto de la apertura que lo mide. |
| `spatial_shift_mode` | Si los cubos se desplazan por **píxeles enteros** o con **submuestreo**. | Un desplazamiento fraccionario interpola, y la interpolación **correlaciona píxeles vecinos**: ese es el origen físico de que el ruido de una apertura no sea la suma en cuadratura de los píxeles. |
| `covariance_factor_box3` | Cuánto se subestima σ en una caja 3×3 si se supone que los píxeles son independientes. | Es la traducción numérica de lo anterior y entra en todo el modelo de ruido ([`docs/noise_model.md`](../../docs/noise_model.md)). |
| `finite_fraction` | Fracción de vóxeles con dato (no NaN) tras alinear y recortar. | Los bordes pierden cobertura al desplazar; si cae mucho, el recorte se comió campo útil. |
| `crop` / `crop_bounds_per_cube` | La ventana espacial que se conserva. | Define el campo donde existen controles al mismo radio que el compañero. |
| `equivalence` | Comparación contra el producto de referencia (histórico o ADP). | Es la prueba de que re-alinear no cambió el dato, solo su rejilla. |


## Resultados que llevaron a la conclusión

Campos clave del `stage01_qc.json`: centrado, shift, crop, STAT propagado y covarianza.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('B1', 'stages/stage01_qc.json'):
        q = nb.load_qc('stages/stage01_qc.json', RUN_ID)
        c = q['crop']
        print('entrada:', q['inputs'][0]['file'])
        print(f"n_cubes = {q['n_cubes']}  (1 = cubo ya combinado en A1; alineación inter-exp = plan B)")
        print(f"centrado: {q['centering_method']} (perfil {q['profile']}, fallback={q['shifts'][0].get('fallback_used')})")
        print(f"centro (y,x) = ({c['center_yx'][0]:.2f}, {c['center_yx'][1]:.2f})  [{c['center_source']}]")
        sh = q['spatial_shifts'][0]
        print(f"shift subpixel (y,x) = ({sh['shift_y']}, {sh['shift_x']})  modo={q['spatial_shift_mode']}")
        print(f"crop = {c['npix']}px  ->  cube_shape {q['cube_shape']}")
        print(f"fracción finita = {q['finite_fraction_per_cube'][0]:.3f}")
        st = q['stat']
        print(f"STAT propagado: {st['propagated']} (kernel {st['interp_kernel']}); "
              f"covarianza box3 = {st['covariance_factor_box3']:.2f}×  <-- nexo con M5/G1")


## Plot — centrado + crop sobre el campo completo

**FITS usado:** `cube_telcorr.fits` (entrada de B1), luz-blanca. Marca el **centro de la estrella** (rojo) y las dos cajas: el **crop final de 170 px** (cyan) y el crop inicial de 80 px (naranja). El halo AO de la primaria domina el campo — por eso el crop de 170 px conserva el halo hasta la posición del compañero.


In [ ]:
try:
    MAKE_PLOT = True   # carga el cubo (~3.3 GB) vía astropy; requiere kernel MUSE
    if MAKE_PLOT:
        try:
            import numpy as np
            import matplotlib.pyplot as plt
            from matplotlib.patches import Rectangle
            from astropy.io import fits

            q = nb.load_qc('stages/stage01_qc.json', RUN_ID)
            cy, cx = q['crop']['center_yx']
            cb = q['crop_bounds_per_cube'][0]; ib = q['initial_crop_bounds_per_cube'][0]
            cube_path = q['inputs'][0]['file']
            print('FITS usado:', cube_path)

            h = fits.open(cube_path, memmap=True)
            wl = np.nanmedian(np.asarray(h[1].data, dtype=np.float32), axis=0); h.close()

            fig, ax = plt.subplots(figsize=(6.2, 6))
            ax.imshow(np.log10(np.clip(wl, 1, None)), origin='lower', cmap='gray')
            ax.plot(cx, cy, '+', color='tab:red', ms=14, mew=2,
                    label=f'centro estrella ({cx:.1f},{cy:.1f})')
            ax.add_patch(Rectangle((cb['x1'], cb['y1']), cb['x2'] - cb['x1'], cb['y2'] - cb['y1'],
                         fill=False, ec='tab:cyan', lw=2, label=f"crop {q['crop_npix']}px"))
            ax.add_patch(Rectangle((ib['x1'], ib['y1']), ib['x2'] - ib['x1'], ib['y2'] - ib['y1'],
                         fill=False, ec='tab:orange', lw=1.2, ls='--',
                         label=f"crop inicial {ib['x2'] - ib['x1']}px"))
            ax.set_title('B1 · centrado + crop sobre luz-blanca (log)')
            ax.legend(fontsize=8, loc='upper right'); ax.axis('off'); fig.tight_layout()
            outdir = nb.run_dir(RUN_ID) / 'plots' / 'b1_align'; outdir.mkdir(parents=True, exist_ok=True)
            fig.savefig(outdir / 'crop.png', dpi=110); print('figura ->', outdir / 'crop.png'); plt.show()
        except Exception as e:
            print('No se pudo generar el plot:', type(e).__name__, e)
            print('Necesita el kernel MUSE (astropy) y el cubo en disco.')
except FileNotFoundError as e:
    print('[etapa pendiente para este objeto]', e)


## Decisiones y notas
- **Migración por equivalencia (B1a):** `run_stage01()` reproduce numéricamente el baseline del notebook histórico; STAT propagado y shifts registrados en QC (B1b).
- **Centrado `maoppy_refined`** (centroide de la PSF física AO, fallback usado); crop de 170 px alrededor del centro medido para conservar el halo hasta el compañero.
- **Covarianza box3 ≈ 6.4×** registrada al propagar STAT: es el remuestreo que correlaciona el ruido → raíz física de M5 (STAT subestima) y de la inflación de apertura (G1). · [`docs/noise_model.md`](../docs/noise_model.md)
- **`n_cubes = 1`:** la alineación entre exposiciones ya se hizo en A1 (plan B); el shift de B1 es 0 → B1 no añade remuestreo nuevo, solo centra/recorta.


## Conclusión (registrada)

**B1: cubo centrado en (166.0, 167.8) y recortado a 170 px; STAT propagado; sin open_issues.**

- **Fecha:** run realineado (cubo `cube_telcorr.fits`, 2026-07-08).
- **Entrada:** `cube_telcorr.fits` (realineado); **salida:** `stage01` stack 170×170 + `stage01_qc.json`.
- **Centrado:** `maoppy_refined`, centro medido (166.0, 167.8); **shift = 0** (`n_cubes = 1`, alineación ya en A1).
- **Fracción finita:** 0.94; **crop:** 170 px (crop inicial 80 px con guarda de 12 px).
- **STAT:** propagado con kernel `bilinear_kernel_squared`; **covarianza box3 = 6.38×** → el nexo cuantificado con M5/G1 y la regla de σ empírico.
- **Downstream:** el cubo de B1 alimenta B2, B3 y C1.
